In [10]:
import os

os.getcwd()

'c:\\Users\\Admin\\release\\czech-pub\\development\\notebooks'

In [11]:
%cd ../..


c:\Users\Admin\release\czech-pub


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [12]:
from utils.helpers import load_dataset
from utils.randomizer import (
    validate_dataset,
    shuffle_options_in_item,
    randomize_dataset,
    verify_randomized_dataset,
    export_jsonl,
)

import json

In [13]:
INPUT_PATH = r"development\data\eval_general\eval\not_randomized\eval_czech_pub.json"
OUTPUT_JSON_PATH = r"development\data\eval_general\eval\randomized\eval_czech_pub.json" 
OUTPUT_JSONL_PATH = r"ddevelopment\data\eval_general\eval\randomized\eval_czech_pub.jsonl"

MASTER_SEED = 42
CORRECT_LABEL_FIELD = "gold_label"   # or "correct_option" if your dataset still uses that

In [14]:
items = load_dataset(INPUT_PATH)

print(f"loaded {len(items)} items.")
print(f"top-level type: {type(items).__name__}")
print(f"first item id: {items[0]['id']}")

loaded 1920 items.
top-level type: list
first item id: implicature-indirect-answer-5


In [15]:
# validating if the dataset structure correspods to the needed standardized structure
errors = validate_dataset(items, correct_label_field=CORRECT_LABEL_FIELD)

print(f"Validation errors found: {len(errors)}")

if errors:
    print("\nFirst 20 errors:")
    for err in errors[:20]:
        print("-", err)
else:
    print("Dataset validation passed.")

Validation errors found: 0
Dataset validation passed.


## testing answer options shuffle on a single item

In [18]:
# before shuffling
sample_item = items[0]

print("ID:", sample_item["id"])
print("gold label:", sample_item[CORRECT_LABEL_FIELD])
print("\noptions before randomization:")

for opt in sample_item["options"]:
    marker = " <-- GOLD LABEL" if opt["label"] == sample_item[CORRECT_LABEL_FIELD] else ""
    print(f"{opt['label']}. [{opt['type']}] {opt['text']}{marker}")

ID: implicature-indirect-answer-5
gold label: B

options before randomization:
A. [direct_answer_positive] Ano, čtu rád motivační knížky.
B. [direct_answer_negative] Ne, nečtu rád motivační knížky. <-- GOLD LABEL
C. [distractor_lexical] Zrovna jejich názvy si hledám v online katalogu.
D. [distractor_associative] Často je vídám ve výloze.


In [19]:
# after shuffling
shuffled_sample = shuffle_options_in_item(
    sample_item,
    master_seed=MASTER_SEED,
    correct_label_field=CORRECT_LABEL_FIELD,
)

print("ID:", shuffled_sample["id"])
print("gold label after shuffle:", shuffled_sample[CORRECT_LABEL_FIELD])
print("\noptions after randomization:")

for opt in shuffled_sample["options"]:
    marker = " <-- GOLD" if opt["label"] == shuffled_sample[CORRECT_LABEL_FIELD] else ""
    print(f"{opt['label']}. [{opt['type']}] {opt['text']}{marker}")

ID: implicature-indirect-answer-5
gold label after shuffle: C

options after randomization:
A. [distractor_lexical] Zrovna jejich názvy si hledám v online katalogu.
B. [distractor_associative] Často je vídám ve výloze.
C. [direct_answer_negative] Ne, nečtu rád motivační knížky. <-- GOLD
D. [direct_answer_positive] Ano, čtu rád motivační knížky.


## randomizing

In [20]:
# now we can randomize the whole dataset (shuffle items + opions in each item) and see the result
randomized_items = randomize_dataset(
    items,
    master_seed=MASTER_SEED,
    correct_label_field=CORRECT_LABEL_FIELD,
)

print(f"original dataset size: {len(items)}")
print(f"randomized dataset size: {len(randomized_items)}")

original dataset size: 1920
randomized dataset size: 1920


## observing randomization results + verification

In [21]:
print("first 10 original item IDs:")
for item in items[:10]:
    print(item["id"])

print("\nfirst 10 randomized item IDs:")
for item in randomized_items[:10]:
    print(item["id"])

first 10 original item IDs:
implicature-indirect-answer-5
implicature-indirect-answer-8
implicature-indirect-answer-23
implicature-indirect-answer-55
implicature-indirect-answer-76
implicature-indirect-answer-81
implicature-indirect-answer-103
implicature-indirect-answer-109
implicature-indirect-answer-129
implicature-indirect-answer-147

first 10 randomized item IDs:
presupposition-change-of-state-4
presupposition-implicative-46
implicature-indirect-answer-406
information-structure-explicit_question-32-focus_object
presupposition-possessive-47
information-structure-correction-51-focus_subject
information-structure-correction-80-focus_object
implicature-indirect-answer-351
information-structure-explicit_question-15-focus_verb
information-structure-explicit_question-33-focus_subject


In [22]:
# inspect troubles
problems = verify_randomized_dataset(
    items,
    randomized_items,
    correct_label_field=CORRECT_LABEL_FIELD,
)

print(f"verification problems found: {len(problems)}")

if problems:
    print("\nfirst 20 problems:")
    for problem in problems[:20]:
        print("-", problem)
else:
    print("post-randomization verification passed.")

verification problems found: 0
post-randomization verification passed.


In [23]:
# quick check to verify reproducibility: randomizing with the same seed returns the same result
randomized_items_again = randomize_dataset(
    items,
    master_seed=MASTER_SEED,
    correct_label_field=CORRECT_LABEL_FIELD,
)

print("verification: identical with same seed:", randomized_items == randomized_items_again)

verification: identical with same seed: True


In [25]:
sample_id = items[777]["id"]

orig_by_id = {item["id"]: item for item in items}
rand_by_id = {item["id"]: item for item in randomized_items}

orig_item = orig_by_id[sample_id]
rand_item = rand_by_id[sample_id]

print("ORIGINAL")
print("ID:", orig_item["id"])
print("Gold label:", orig_item[CORRECT_LABEL_FIELD])
for opt in orig_item["options"]:
    marker = " <-- GOLD" if opt["label"] == orig_item[CORRECT_LABEL_FIELD] else ""
    print(f"{opt['label']}. [{opt['type']}] {opt['text']}{marker}")

print("\nRANDOMIZED")
print("ID:", rand_item["id"])
print("Gold label:", rand_item[CORRECT_LABEL_FIELD])
for opt in rand_item["options"]:
    marker = " <-- GOLD" if opt["label"] == rand_item[CORRECT_LABEL_FIELD] else ""
    print(f"{opt['label']}. [{opt['type']}] {opt['text']}{marker}")

ORIGINAL
ID: information-structure-baseline-60-focus_object
Gold label: A
A. [object_last] Alena rozbila talíř. <-- GOLD
B. [subject_last] Talíř rozbila Alena.
C. [verb_last] Alena talíř rozbila.
D. [distractor] Alena utřela stůl.

RANDOMIZED
ID: information-structure-baseline-60-focus_object
Gold label: B
A. [verb_last] Alena talíř rozbila.
B. [object_last] Alena rozbila talíř. <-- GOLD
C. [distractor] Alena utřela stůl.
D. [subject_last] Talíř rozbila Alena.


## save results to files

In [ ]:
with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(randomized_items, f, ensure_ascii=False, indent=2)

print(f"Saved randomized JSON to: {OUTPUT_JSON_PATH}")
print(f"Master seed: {MASTER_SEED}")

In [ ]:
export_jsonl(randomized_items, OUTPUT_JSONL_PATH)
print(f"Saved randomized JSONL to: {OUTPUT_JSONL_PATH}")